In [1]:
import asyncio
import time
import aiohttp

In [2]:
async def say_after(delay, what):
    print(f"start say_after {locals()=}")
    time.sleep(delay)
    print(what)


async def main():
    t1 = time.time()

    task1 = asyncio.create_task(say_after(1, "hello"))
    print("after say 1")
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)

    t2 = time.time()
    print("Total time:", t2 - t1)


await main()

after say 1
start say_after locals()={'delay': 1, 'what': 'hello'}
hello
start say_after locals()={'delay': 2, 'what': 'world'}
world
Total time: 3.010138988494873


In [7]:
def async_timeit(coro):
    async def inner(*args, **kwargs):
        t1 = time.time()
        try:
            return await coro(*args, **kwargs)
        finally:
            t2 = time.time()
            print(f"total time of `{coro.__name__}`:", t2 - t1)
    return inner

In [8]:
async def say_after(delay, what):
    print(f"start say_after {locals()=}")
    time.sleep(delay)
    print(what)


@async_timeit
async def main():
    task1 = asyncio.create_task(say_after(1, "hello"))
    print("after say 1")
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)


await main()

after say 1
start say_after locals()={'delay': 1, 'what': 'hello'}
hello
start say_after locals()={'delay': 2, 'what': 'world'}
world
total time of `main`: 3.013068199157715


In [11]:
async def say_after(delay, what):
    print(f"start say_after {locals()=}")
    await asyncio.to_thread(time.sleep, delay)
    print(what)


@async_timeit
async def main():
    task1 = asyncio.create_task(say_after(1, "hello"))
    print("after say 1")
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)


await main()

after say 1
start say_after locals()={'delay': 1, 'what': 'hello'}
start say_after locals()={'delay': 2, 'what': 'world'}
hello
world
total time of `main`: 2.006664276123047


In [13]:
async def say_after(delay, what):
    print(f"start say_after {locals()=}")
    for i in range(10):
        time.sleep(delay / 10)
        await asyncio.sleep(0)
        print("coro", what, i)
    print(what)


@async_timeit
async def main():
    task1 = asyncio.create_task(say_after(1, "hello"))
    print("after say 1")
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)


await main()

after say 1
start say_after locals()={'delay': 1, 'what': 'hello'}
start say_after locals()={'delay': 2, 'what': 'world'}
coro hello 0
coro world 0
coro hello 1
coro world 1
coro hello 2
coro world 2
coro hello 3
coro world 3
coro hello 4
coro world 4
coro hello 5
coro world 5
coro hello 6
coro world 6
coro hello 7
coro world 7
coro hello 8
coro world 8
coro hello 9
hello
coro world 9
world
total time of `main`: 3.0626847743988037


In [14]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"


async def fetch_url(url, i):
    print("start fetch", i)
    async with aiohttp.ClientSession() as session:
        async with session.get(URL) as resp:
            await resp.text()
    print("finish fetch", i)


async def fetch_batch_urls(urls):
    tasks = []
    for i, url in enumerate(urls):
        tasks.append(fetch_url(url, i))

    await asyncio.gather(*tasks)


async def run():
    t1 = time.time()
    urls = [URL] * 10

    res = await fetch_batch_urls(urls)
    t2 = time.time()
    print("total time", t2 - t1)


await run()

start fetch 0
start fetch 1
start fetch 2
start fetch 3
start fetch 4
start fetch 5
start fetch 6
start fetch 7
start fetch 8
start fetch 9
finish fetch 4
finish fetch 0
finish fetch 8
finish fetch 5
finish fetch 1
finish fetch 9
finish fetch 6
finish fetch 3
finish fetch 7
finish fetch 2
total time 1.047088861465454


In [18]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
WORKERS = 10


async def fetch_url(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(URL) as resp:
            await resp.text()


async def worker(que, worker_name):
    print(f"start worker", worker_name)
    cnt = 0

    while True:
        url = await que.get()
        if url is None:
            await que.put(url)
            break

        res = await fetch_url(url)
        cnt += 1

    print(f"finish worker", worker_name, cnt)


async def run():
    t1 = time.time()
    urls = [URL] * 30
    
    que = asyncio.Queue()
    for url in urls:
        await que.put(url)
    await que.put(None)

    workers = [
        asyncio.create_task(worker(que, f"worker_{i}"))
        for i in range(WORKERS)
    ]
    await asyncio.gather(*workers)

    t2 = time.time()
    print("total time", t2 - t1)


await run()

start worker worker_0
start worker worker_1
start worker worker_2
start worker worker_3
start worker worker_4
start worker worker_5
start worker worker_6
start worker worker_7
start worker worker_8
start worker worker_9
finish worker worker_4 3
finish worker worker_7 3
finish worker worker_9 3
finish worker worker_3 3
finish worker worker_5 3
finish worker worker_0 3
finish worker worker_1 3
finish worker worker_6 3
finish worker worker_2 3
finish worker worker_8 3
total time 2.264832019805908


In [22]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
WORKERS = 10


async def fetch_url(url, session):
    async with session.get(URL) as resp:
        await resp.text()


@async_timeit
async def worker(que, session, worker_name):
    print(f"start worker", worker_name)
    cnt = 0

    while True:
        url = await que.get()
        if url is None:
            await que.put(url)
            break

        res = await fetch_url(url, session)
        cnt += 1

    print(f"finish worker", worker_name, cnt)


@async_timeit
async def run():
    urls = [URL] * 30
    
    que = asyncio.Queue()
    for url in urls:
        await que.put(url)
    await que.put(None)

    async with aiohttp.ClientSession() as session:
        workers = [
            asyncio.create_task(worker(que, session, f"worker_{i}"))
            for i in range(WORKERS)
        ]
        await asyncio.gather(*workers)


await run()

start worker worker_0
start worker worker_1
start worker worker_2
start worker worker_3
start worker worker_4
start worker worker_5
start worker worker_6
start worker worker_7
start worker worker_8
start worker worker_9
finish worker worker_9 2
total time of `worker`: 1.0363030433654785
finish worker worker_7 1
total time of `worker`: 1.0382850170135498
finish worker worker_2 2
total time of `worker`: 1.066810131072998
finish worker worker_1 4
total time of `worker`: 1.0901119709014893
finish worker worker_0 3
total time of `worker`: 1.103935956954956
finish worker worker_8 4
total time of `worker`: 1.1141839027404785
finish worker worker_3 4
total time of `worker`: 1.1583080291748047
finish worker worker_6 4
total time of `worker`: 1.168335199356079
finish worker worker_5 4
total time of `worker`: 1.1788489818572998
finish worker worker_4 2
total time of `worker`: 1.1925239562988281
total time of `run`: 1.1960780620574951


In [31]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
WORKERS = 10


async def fetch_url(url, session, sem):
    async with sem:
        #await asyncio.sleep(0.1)
        async with session.get(URL) as resp:
            await resp.text()


@async_timeit
async def worker(que, session, sem, worker_name):
    print(f"start worker", worker_name)
    cnt = 0

    while True:
        url = await que.get()
        if url is None:
            await que.put(url)
            break

        res = await fetch_url(url, session, sem)
        cnt += 1

    print(f"finish worker", worker_name, cnt)


@async_timeit
async def run():
    urls = [URL] * 30
    
    que = asyncio.Queue()
    for url in urls:
        await que.put(url)
    await que.put(None)

    sem = asyncio.Semaphore(5)

    async with aiohttp.ClientSession() as session:
        workers = [
            asyncio.create_task(worker(que, session, sem, f"worker_{i}"))
            for i in range(WORKERS)
        ]
        await asyncio.gather(*workers)


await run()

start worker worker_0
start worker worker_1
start worker worker_2
start worker worker_3
start worker worker_4
start worker worker_5
start worker worker_6
start worker worker_7
start worker worker_8
start worker worker_9
finish worker worker_2 3
total time of `worker`: 0.9733400344848633
finish worker worker_0 3
total time of `worker`: 0.979762077331543
finish worker worker_1 3
total time of `worker`: 1.0043301582336426
finish worker worker_5 3
total time of `worker`: 1.0145421028137207
finish worker worker_4 3
total time of `worker`: 1.0327582359313965
finish worker worker_7 3
total time of `worker`: 1.0451679229736328
finish worker worker_8 3
total time of `worker`: 1.0901501178741455
finish worker worker_9 3
total time of `worker`: 1.096625804901123
finish worker worker_3 4
total time of `worker`: 1.1242201328277588
finish worker worker_6 2
total time of `worker`: 1.1797730922698975
total time of `run`: 1.1841411590576172


In [34]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
WORKERS = 10


async def fetch_url(url, session, sem):
    async with sem:
        async with session.get(URL) as resp:
            await resp.text()


@async_timeit
async def fetch_worker(que, session, sem, worker_name):
    print(f"start worker", worker_name)
    cnt = 0

    while True:
        url = await que.get()

        res = await fetch_url(url, session, sem)
        que.task_done()
        cnt += 1

    print(f"finish worker", worker_name, cnt)


@async_timeit
async def run():
    urls = [URL] * 30
    
    que = asyncio.Queue()
    for url in urls:
        await que.put(url)

    sem = asyncio.Semaphore(5)

    async with aiohttp.ClientSession() as session:
        workers = [
            asyncio.create_task(fetch_worker(que, session, sem, f"worker_{i}"))
            for i in range(WORKERS)
        ]
        await que.join()

        for worker in workers:
            worker.cancel()


await run()

start worker worker_0
start worker worker_1
start worker worker_2
start worker worker_3
start worker worker_4
start worker worker_5
start worker worker_6
start worker worker_7
start worker worker_8
start worker worker_9
total time of `fetch_worker`: 4.5877251625061035
total time of `fetch_worker`: 4.587673902511597
total time of `fetch_worker`: 4.58765721321106
total time of `fetch_worker`: 4.587646961212158
total time of `fetch_worker`: 4.587642192840576
total time of `fetch_worker`: 4.587635040283203
total time of `fetch_worker`: 4.5876500606536865
total time of `fetch_worker`: 4.587665796279907
total time of `fetch_worker`: 4.587682247161865
total time of `fetch_worker`: 4.587697267532349
total time of `run`: 4.58960223197937


In [36]:
async def counter(n):
    for i in range(n):
        yield i
        print(f"yield {i=}")
        await asyncio.sleep(0.1)


async for x in counter(5):
    print(f"{x=}")

x=0
yield i=0
x=1
yield i=1
x=2
yield i=2
x=3
yield i=3
x=4
yield i=4


In [41]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"


@async_timeit
async def fetch_url(url, i, session):
    print("start fetch", i)
    async with session.get(URL) as resp:
        await resp.text()
    print("finish fetch", i)


async def fetch_batch_urls(urls):
    tasks = []
    async with aiohttp.ClientSession() as session:
        for i, url in enumerate(urls):
            tasks.append(fetch_url(url, i, session))

        await asyncio.gather(*tasks)


@async_timeit
async def run():
    urls = [URL] * 10

    res = await fetch_batch_urls(urls)


await run()

start fetch 0
start fetch 1
start fetch 2
start fetch 3
start fetch 4
start fetch 5
start fetch 6
start fetch 7
start fetch 8
start fetch 9
finish fetch 8
total time of `fetch_url`: 0.7252218723297119
finish fetch 3
total time of `fetch_url`: 0.7289738655090332
finish fetch 2
total time of `fetch_url`: 0.7696762084960938
finish fetch 9
total time of `fetch_url`: 0.8965940475463867
finish fetch 0
total time of `fetch_url`: 0.9010131359100342
finish fetch 7
total time of `fetch_url`: 0.9557929039001465
finish fetch 5
total time of `fetch_url`: 1.0239150524139404
finish fetch 1
total time of `fetch_url`: 1.033430814743042
finish fetch 4
total time of `fetch_url`: 1.06803297996521
finish fetch 6
total time of `fetch_url`: 1.073930025100708
total time of `run`: 1.0779612064361572


In [44]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"


@async_timeit
async def fetch_url(url, name, session):
    print("start fetch", name)
    async with session.get(URL) as resp:
        text = await resp.text()
    print("finish fetch", name)
    return (resp.status, len(text), name)


async def fetch_batch_urls(urls):
    tasks = []
    async with aiohttp.ClientSession() as session:
        for i, url in enumerate(urls):
            tasks.append(fetch_url(url, f"fetch_{i}", session))

        start = time.time()
        for compl_task in asyncio.as_completed(tasks):
            print("compl_task", time.time() - start)
            task_result = await compl_task
            print("task_result", time.time() - start)


@async_timeit
async def run():
    urls = [URL] * 10

    res = await fetch_batch_urls(urls)


await run()

compl_task 8.511543273925781e-05
start fetch fetch_1
start fetch fetch_3
start fetch fetch_5
start fetch fetch_7
start fetch fetch_0
start fetch fetch_2
start fetch fetch_4
start fetch fetch_6
start fetch fetch_8
start fetch fetch_9
finish fetch fetch_3
total time of `fetch_url`: 0.5594489574432373
task_result 0.5609710216522217
compl_task 0.5610013008117676
finish fetch fetch_2
total time of `fetch_url`: 0.6235771179199219
task_result 0.6252481937408447
compl_task 0.6252791881561279
finish fetch fetch_7
total time of `fetch_url`: 0.7404019832611084
task_result 0.7419960498809814
compl_task 0.742011308670044
finish fetch fetch_8
total time of `fetch_url`: 0.7556629180908203
task_result 0.7573282718658447
compl_task 0.7573611736297607
finish fetch fetch_0
total time of `fetch_url`: 0.7622220516204834
finish fetch fetch_9
total time of `fetch_url`: 0.7622280120849609
task_result 0.7640702724456787
compl_task 0.7640841007232666
task_result 0.7640933990478516
compl_task 0.7641003131866455


In [45]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"


@async_timeit
async def fetch_url(url, name, session):
    print("start fetch", name)
    async with session.get(URL) as resp:
        text = await resp.text()
    print("finish fetch", name)
    return (resp.status, len(text), name)


async def fetch_batch_urls(urls):
    tasks = []
    async with aiohttp.ClientSession() as session:
        for i, url in enumerate(urls):
            tasks.append(fetch_url(url, f"fetch_{i}", session))

        start = time.time()
        async for compl_task in asyncio.as_completed(tasks):
            print("compl_task", time.time() - start)
            task_result = await compl_task
            print("task_result", time.time() - start)


@async_timeit
async def run():
    urls = [URL] * 10

    res = await fetch_batch_urls(urls)


await run()

start fetch fetch_9
start fetch fetch_1
start fetch fetch_3
start fetch fetch_5
start fetch fetch_0
start fetch fetch_2
start fetch fetch_4
start fetch fetch_6
start fetch fetch_8
start fetch fetch_7
finish fetch fetch_0
total time of `fetch_url`: 0.625877857208252
compl_task 0.627722978591919
task_result 0.6277530193328857
finish fetch fetch_2
total time of `fetch_url`: 0.6410582065582275
compl_task 0.6434829235076904
task_result 0.6435279846191406
finish fetch fetch_3
total time of `fetch_url`: 0.6867241859436035
compl_task 0.688018798828125
task_result 0.688060998916626
finish fetch fetch_1
total time of `fetch_url`: 0.706489086151123
compl_task 0.7078378200531006
task_result 0.707866907119751
finish fetch fetch_9
total time of `fetch_url`: 0.7322590351104736
compl_task 0.7328157424926758
task_result 0.7328298091888428
finish fetch fetch_4
total time of `fetch_url`: 0.7400619983673096
compl_task 0.7416229248046875
task_result 0.7416329383850098
finish fetch fetch_7
total time of `fe

In [48]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"


async def fetch_url(url, name, session):
    async with session.get(URL) as resp:
        text = await resp.text()
    return (resp.status, len(text), name)


async def fetch_batch_urls(urls):
    tasks = []
    async with aiohttp.ClientSession() as session:
        for i, url in enumerate(urls):
            tasks.append(fetch_url(url, f"fetch_{i}", session))

        async for compl_task in asyncio.as_completed(tasks):
            task_result = await compl_task

            name = task_result[2]
            idx = int(name.split("_")[1])
            if idx % 2 == 0:
                yield task_result


@async_timeit
async def run():
    urls = [URL] * 10

    async for res in fetch_batch_urls(urls):
        print(res)


await run()

(200, 554835, 'fetch_6')
(200, 554835, 'fetch_2')
(200, 554835, 'fetch_4')
(200, 554835, 'fetch_8')
(200, 554835, 'fetch_0')
total time of `run`: 0.8946249485015869


In [63]:
async def update_user_data(data, delay):
    print(f"start update_user_data {locals()=}")
    await asyncio.sleep(delay)
    print(f"finish update_user_data {locals()=}")


@async_timeit
async def main():
    task = asyncio.create_task(update_user_data("data", 2))

    await task
    print("finish main")


main_task = asyncio.create_task(main())
await main_task

start update_user_data locals()={'data': 'data', 'delay': 2}
finish update_user_data locals()={'data': 'data', 'delay': 2}
finish main
total time of `main`: 2.0019750595092773


In [64]:
@async_timeit
async def update_user_data(data, delay):
    print(f"start update_user_data {locals()=}")
    await asyncio.sleep(delay)
    print(f"finish update_user_data {locals()=}")


@async_timeit
async def main():
    task = asyncio.create_task(update_user_data("data", 20))

    await task
    print("finish main")


main_task = asyncio.create_task(main())

start update_user_data locals()={'data': 'data', 'delay': 20}
total time of `update_user_data`: 5.116658926010132
total time of `main`: 5.1169350147247314


In [65]:
main_task

<Task pending name='Task-1432' coro=<async_timeit.<locals>.inner() running at /var/folders/j3/m947cq0j0fx8t0rrdzw5kmb00000gp/T/ipykernel_78763/1789918392.py:5> wait_for=<Task pending name='Task-1433' coro=<async_timeit.<locals>.inner() running at /var/folders/j3/m947cq0j0fx8t0rrdzw5kmb00000gp/T/ipykernel_78763/1789918392.py:5> wait_for=<Future pending cb=[Task.task_wakeup()]> cb=[Task.task_wakeup()]>>

In [66]:
main_task.cancelled()

False

In [67]:
main_task.cancel()

True

In [68]:
main_task.cancelled()

True

In [69]:
@async_timeit
async def update_user_data(data, delay):
    print(f"start update_user_data {locals()=}")
    await asyncio.sleep(delay)
    print(f"finish update_user_data {locals()=}")


@async_timeit
async def main():
    task = asyncio.shield(update_user_data("data", 20))

    await task
    print("finish main")


main_task = asyncio.create_task(main())

start update_user_data locals()={'data': 'data', 'delay': 20}
total time of `main`: 6.818912029266357
finish update_user_data locals()={'data': 'data', 'delay': 20}
total time of `update_user_data`: 20.001054048538208


In [70]:
main_task.cancel()

True

In [71]:
main_task.cancelled()

True

In [73]:
@async_timeit
async def update_user_data(data, delay):
    print(f"start update_user_data {locals()=}")
    await asyncio.sleep(delay // 2)

    if data == "data_2":
        raise ValueError("wrong data")

    await asyncio.sleep(delay // 2)

    print(f"finish update_user_data {locals()=}")


@async_timeit
async def main():
    task1 = asyncio.create_task(update_user_data("data_1", 4))
    task2 = asyncio.create_task(update_user_data("data_2", 4))
    task3 = asyncio.create_task(update_user_data("data_3", 4))

    await asyncio.gather(task1, task2, task3)
    print("finish main")


await main()

start update_user_data locals()={'data': 'data_1', 'delay': 4}
start update_user_data locals()={'data': 'data_2', 'delay': 4}
start update_user_data locals()={'data': 'data_3', 'delay': 4}
total time of `update_user_data`: 2.0011229515075684
total time of `main`: 2.0031049251556396


ValueError: wrong data

finish update_user_data locals()={'data': 'data_1', 'delay': 4}
total time of `update_user_data`: 4.002936840057373
finish update_user_data locals()={'data': 'data_3', 'delay': 4}
total time of `update_user_data`: 4.0021069049835205


In [74]:
@async_timeit
async def update_user_data(data, delay):
    print(f"start update_user_data {locals()=}")
    await asyncio.sleep(delay // 2)

    if data == "data_2":
        raise ValueError("wrong data")

    await asyncio.sleep(delay // 2)

    print(f"finish update_user_data {locals()=}")


@async_timeit
async def main():

    async with asyncio.TaskGroup() as group_tasks:
        task1 = group_tasks.create_task(update_user_data("data_1", 4))
        #task2 = group_tasks.create_task(update_user_data("data_2", 4))
        task3 = group_tasks.create_task(update_user_data("data_3", 4))

    print("finish main")


await main()

start update_user_data locals()={'data': 'data_1', 'delay': 4}
start update_user_data locals()={'data': 'data_3', 'delay': 4}
finish update_user_data locals()={'data': 'data_1', 'delay': 4}
total time of `update_user_data`: 4.0026328563690186
finish update_user_data locals()={'data': 'data_3', 'delay': 4}
total time of `update_user_data`: 4.0023839473724365
finish main
total time of `main`: 4.003277063369751


In [84]:
@async_timeit
async def update_user_data(data, delay):
    print(f"start update_user_data {locals()=}")
    await asyncio.sleep(delay // 2)

    if data == "data_2":
        raise ValueError("wrong data")

    await asyncio.sleep(delay // 2)

    print(f"finish update_user_data {locals()=}")
    return data


@async_timeit
async def main():

    try:
        async with asyncio.TaskGroup() as group_tasks:
            task1 = group_tasks.create_task(update_user_data("data_1", 4))
            task2 = group_tasks.create_task(update_user_data("data_2", 4))
            task3 = group_tasks.create_task(update_user_data("data_3", 4))
    except Exception as err:
        print("ERROR:", err)

    print("after TaskGroup")

    for task in (task1, task2, task3):
        try:
            print(f"Task {task.get_name()}, {task.cancelled(), {task.result()}}")
        except BaseException as err:
            print("task error", err, type(err))

    print("finish main")


await main()

start update_user_data locals()={'data': 'data_1', 'delay': 4}
start update_user_data locals()={'data': 'data_2', 'delay': 4}
start update_user_data locals()={'data': 'data_3', 'delay': 4}
total time of `update_user_data`: 2.001122236251831
total time of `update_user_data`: 2.0015861988067627
total time of `update_user_data`: 2.002274990081787
ERROR: unhandled errors in a TaskGroup (1 sub-exception)
after TaskGroup
task error  <class 'asyncio.exceptions.CancelledError'>
task error wrong data <class 'ValueError'>
task error  <class 'asyncio.exceptions.CancelledError'>
finish main
total time of `main`: 2.0027520656585693


In [85]:
issubclass(asyncio.exceptions.CancelledError, Exception)

False

In [86]:
issubclass(asyncio.exceptions.CancelledError, BaseException)

True

In [87]:
import unittest
from unittest import mock

In [96]:
class TestFetch(unittest.TestCase):
    async def test_fetch(self):
        self.assertEqual(99, 9)


unittest.main(argv=[''], verbosity=2, exit=False)

test_fetch (__main__.TestFetch.test_fetch) ... /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/unittest/case.py:725: DeprecationWarning: It is deprecated to return a value that is not None from a test case (<bound method TestFetch.test_fetch of <__main__.TestFetch testMethod=test_fetch>> returned 'coroutine'). Maybe you forgot to use IsolatedAsyncioTestCase as the base class?
  return self.run(*args, **kwds)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/unittest/case.py:669: RuntimeWarning: coroutine 'TestFetch.test_fetch' was never awaited
  self._callTestMethod(testMethod)
ok

----------------------------------------------------------------------
Ran 1 test in 0.001s

OK
